Perform exploratory data analysis

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import ast
from sklearn import preprocessing
import seaborn as sns
from collections import Counter

# import model_training as mt
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
new_df = pd.read_csv("data\prepared_data.csv")
print(new_df.info())

### EDA on dataset level

##### Propaganda technique distribution in dataset

In [ ]:
def prettify_label(label):
    words = re.findall(r'[A-Z]?[a-z]+', label)
    words = [word.capitalize() for word in words if word]
    return ' '.join(words)

new_df['pretty_label'] = new_df['label'].apply(prettify_label)
new_df = new_df[new_df["fragment"].str.strip() != ""]

In [ ]:
def label_encoder(df: pd.DataFrame, label_column: str):
    encoder = preprocessing.LabelEncoder()
    df["encoded_labels"] = encoder.fit_transform(df[label_column])
    return df

In [ ]:
new_df["pretty_label"].value_counts(ascending=True).plot.barh()
plt.title("Propaganda technique distribution")
plt.ylabel("")
plt.show()

##### Propaganda fragment lenght distribution in dataset

In [ ]:
# Fragment length in words
fragment_lengths_words = [len(x.split()) for x in new_df["fragment"]]

# 99th percentile of fragment lengths
percentile99 = int(np.percentile(fragment_lengths_words, 99))
print("99th percentile of fragment lengths", percentile99)

plt.hist(fragment_lengths_words, range=(0, percentile99), bins=30, edgecolor="black")
plt.axvline(x=np.mean(fragment_lengths_words), c="navy", ls="--")
plt.xlabel("Fragment length in words")
plt.ylabel("Distribution")
plt.title("Distribution of fragment lengths (in words)")
plt.show()


print("Shortest fragment (words) length:", min(fragment_lengths_words))
print("Longest fragment (words) length:", max(fragment_lengths_words))
print("Average fragment (words) length:", np.mean(fragment_lengths_words))
print("Standard deviation of fragment (words) length:", np.std(fragment_lengths_words))


### EDA on article level
##### Propaganda technique distribution - number of unique techniques per article

In [ ]:
# check how many labels one text has
ids = pd.unique(new_df["id"])
num_labs = []
num_fragments_per_article = []
for id in ids:
    df_id = new_df[new_df["id"] == id]
    num_fragments_per_article.append(len(df_id))
    unique_labs = pd.unique(df_id["label"])
    num_labs.append(len(unique_labs))


plt.hist(num_labs, bins=np.arange(0,10)+0.5, range=[0, 10], edgecolor="black")
plt.axvline(x=np.mean(num_labs), c="navy", ls="--")
plt.xlabel("Number of techniques per article")
plt.ylabel("Number of articles")
plt.title("Distribution of propaganda techniques per article")
plt.show()

print("Average number of techniques per article", np.mean(num_labs))
print("Min number of techniques per article", np.min(num_labs))
print("Max number of techniques per article", np.max(num_labs))


##### Propaganda fragment distribution - number of fragments per article

In [ ]:
percentile99 = int(np.percentile(num_fragments_per_article, 99))
print("99th percentile of fragment lengths", percentile99)

plt.hist(num_fragments_per_article, bins=20, range=[0, percentile99], edgecolor="black")
plt.axvline(x=np.mean(num_fragments_per_article), c="navy", ls="--")
plt.xlabel("Number of fragments per article")
plt.ylabel("Number of articles")
plt.title("Distribution of propaganda fragments per article")
plt.show()

print("Average number of fragments per article", np.mean(num_fragments_per_article))
print("Min number of fragments per article", np.min(num_fragments_per_article))
print("Max number of fragments per article", np.max(num_fragments_per_article))

### EDA on technique level
##### Fragment lengths aggregated by propaganda techniques

In [ ]:
new_df["words_per_fragment"] = new_df["fragment"].str.split().apply(len)
new_df.boxplot("words_per_fragment", by="pretty_label", grid=False,
                  showfliers=False, color="black", vert=False)
plt.suptitle("")
plt.title("Fragment lengths")
plt.xlabel("")
plt.ylabel("")
plt.show()


### Evaluate propaganda technique overlapping


In [ ]:
# get ids, spans, fragments and labels. 
df = pd.read_csv("data\concatenated.csv")
df = df.loc[:, ["annotation_id", "content", "label"]]
df.dropna(inplace=True)

fragments = []
for index, row in df.iterrows():
    content: str = row["content"] # format: string
    id: int = row["annotation_id"]
    labels_list: list = ast.literal_eval(row["label"]) # format: list
    for dictionary in labels_list:
        start = dictionary.get("start")
        end = dictionary.get("end")
        labels = dictionary.get("labels")

        labels = "".join(labels)
        fragments.append((id, content[start:end], labels, start, end))
print(len(fragments))

new_df = pd.DataFrame(data=fragments, columns=["id", "fragment", "label", "start", "end"])

print(new_df.shape)
new_df = new_df[new_df["fragment"].str.strip() != ""] # removing empty fragment strings
new_df = new_df[new_df["label"] != ""] # removing empty labels
print(new_df.shape)

new_df = label_encoder(new_df, "label")
# print(new_df.head())
new_df.to_csv("data\data_start_stop.csv", index=False)

# from collections import Counter
# print(Counter(new_df["label"]))
# Counter(new_df["encoded_labels"])



##### Finding the exact match by naive matching
the matching was done on the exact span basis, not the similarity of text.

In [ ]:
# FIND EXACT MATCH

multilabel = []
only_multilabel = []
ids = pd.unique(new_df["id"])
for id in ids:
    id_df: pd.DataFrame = new_df.loc[new_df["id"] == id]
    
    checked_spans = set()

    for i in range(len(id_df)):
        exact_match_labels = []
        span_start = id_df.iloc[i, 3]
        span_end = id_df.iloc[i, 4]
        span_label = id_df.iloc[i, 5]
        exact_match_labels.append(span_label)

        span_id = (id, span_start, span_end)

        if span_id in checked_spans:
            continue
        checked_spans.add(span_id)
        for j in range(i+1, len(id_df)):
            compare_span_start = id_df.iloc[j, 3]
            compare_span_end = id_df.iloc[j, 4]
            compare_span_label = id_df.iloc[j, 5]

            if span_start == compare_span_start and span_end == compare_span_end:
                if span_label != compare_span_label:
                    exact_match_labels.append(compare_span_label)
        if len(exact_match_labels) > 1:
            only_multilabel.append((id, id_df.iloc[i, 1], exact_match_labels))
        multilabel.append((id, id_df.iloc[i, 1], exact_match_labels))


print("Multilabel dataset:")
print(len(multilabel))

print("ONLY Multilabel dataset:")
print(len(only_multilabel))


multilabel_df = pd.DataFrame(multilabel, columns=["id", "fragment", "labels"])
multilabel_df.to_csv("data\multilabel.csv", index=False)

##### Find out which classes co-occur most often?

In [ ]:
only_multilabel_df = pd.DataFrame(only_multilabel, columns=["id", "fragment", "labels"])
list_labels = only_multilabel_df["labels"].to_list()
matrix = np.zeros(shape=(10, 10))
for i in range(len(list_labels)):
    lst = list_labels[i]
    if len(lst) == 2:
        matrix[lst[0]][lst[1]] += 1
        matrix[lst[1]][lst[0]] += 1
    else:
        matrix[lst[0]][lst[1]] += 1
        matrix[lst[1]][lst[0]] += 1
        matrix[lst[0]][lst[2]] += 1
        matrix[lst[2]][lst[0]] += 1
        matrix[lst[1]][lst[2]] += 1
        matrix[lst[2]][lst[1]] += 1


df_matrix = pd.DataFrame(matrix, columns=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], index=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [ ]:
techs = ['Appeal To Authority', 'Doubt', 'Emotional Expression', 'Following Behind','Reductio Ad Hitlerum', 'Repetition', 'Simplification',  'Uncertainty', 'Waving The Flag', 'Whataboutism/RedHerring/StrawMan', ]
sns.heatmap(df_matrix, cmap="Blues", xticklabels=techs, yticklabels=techs)
plt.title("Co-occurring techniques")
plt.show()
df_matrix

##### Single label dataset

In [ ]:
singlelabel_df = multilabel_df[multilabel_df["labels"].map(len) == 1]
singlelabel_df["label"] = singlelabel_df["labels"].apply(lambda x: np.array(x).astype(int).item())
singlelabel_df.to_csv("data\single_label.csv", index=False)

In [ ]:
singlelabel_df["label"].value_counts(ascending=True).plot.barh()
plt.title("Propaganda technique distribution")
plt.ylabel("")
plt.show()

Counter(singlelabel_df["label"])